In [1]:
from pathlib import Path
from collections import defaultdict
import re
import warnings

import numpy as np
import pandas as pd
import joblib

from scapy.all import rdpcap, IP, TCP, UDP, ICMP, Raw
from scapy.layers.dns import DNS, DNSQR, DNSRR

warnings.filterwarnings("ignore")

PCAP_PATH = Path("Dataset-2.pcap")

MODEL_PATH = Path("artifacts/xgb_nfv2_model.joblib")
ENCODER_PATH = Path("artifacts/label_encoder.joblib")
FEATURE_PATH = Path("artifacts/feature_names.joblib")

OUTPUT_PATH = Path("deneme_nfv2_features.csv")

assert PCAP_PATH.exists(), f"PCAP bulunamadi: {PCAP_PATH}"
assert MODEL_PATH.exists(), f"Model bulunamadi: {MODEL_PATH}"
assert ENCODER_PATH.exists(), f"Encoder bulunamadi: {ENCODER_PATH}"
assert FEATURE_PATH.exists(), f"Feature listesi bulunamadi: {FEATURE_PATH}"

print("Yollar hazir.")

Yollar hazir.


In [2]:
packets = rdpcap(str(PCAP_PATH))

model = joblib.load(MODEL_PATH)
label_encoder = joblib.load(ENCODER_PATH)
feature_names = joblib.load(FEATURE_PATH)

print("Okunan paket:", len(packets))
print("Model feature sayisi:", model.n_features_in_)
print("Kayitli feature sayisi:", len(feature_names))
print("Sinif sayisi:", len(label_encoder.classes_))

assert model.n_features_in_ == 39
assert len(feature_names) == 39

Okunan paket: 365
Model feature sayisi: 39
Kayitli feature sayisi: 39
Sinif sayisi: 16


In [3]:
FTP_REPLY_PATTERN = re.compile(rb"^(\d{3})[\s-]")


def get_transport_info(pkt):
    if TCP in pkt:
        return (
            int(pkt[TCP].sport),
            int(pkt[TCP].dport),
            6
        )

    if UDP in pkt:
        return (
            int(pkt[UDP].sport),
            int(pkt[UDP].dport),
            17
        )

    if ICMP in pkt:
        return 0, 0, 1

    if IP in pkt:
        return 0, 0, int(pkt[IP].proto)

    return 0, 0, 0


def make_forward_key(src, dst, sport, dport, proto):
    return (
        src,
        dst,
        int(sport),
        int(dport),
        int(proto)
    )


def make_reverse_key(src, dst, sport, dport, proto):
    return (
        dst,
        src,
        int(dport),
        int(sport),
        int(proto)
    )


def directional_duration_ms(first_ts, last_ts):
    if first_ts is None or last_ts is None:
        return 0.0

    return max(
        0.0,
        (float(last_ts) - float(first_ts)) * 1000.0
    )


def calculate_second_bytes(byte_count, first_ts, last_ts):
    if byte_count <= 0:
        return 0.0

    duration_ms = directional_duration_ms(
        first_ts,
        last_ts
    )

    # Tek paketli yönlerde 1 ms taban pencere
    effective_seconds = max(
        duration_ms / 1000.0,
        0.001
    )

    return float(byte_count) / effective_seconds


def update_packet_size_bin(flow, packet_size):
    if packet_size <= 128:
        flow["NUM_PKTS_UP_TO_128_BYTES"] += 1

    elif packet_size <= 256:
        flow["NUM_PKTS_128_TO_256_BYTES"] += 1

    elif packet_size <= 512:
        flow["NUM_PKTS_256_TO_512_BYTES"] += 1

    elif packet_size <= 1024:
        flow["NUM_PKTS_512_TO_1024_BYTES"] += 1

    elif packet_size <= 1514:
        flow["NUM_PKTS_1024_TO_1514_BYTES"] += 1


def tcp_sequence_range(pkt):
    if TCP not in pkt:
        return None

    tcp = pkt[TCP]

    payload_length = len(bytes(tcp.payload))
    consumed_sequence = payload_length
    flags = int(tcp.flags)

    # SYN
    if flags & 0x02:
        consumed_sequence += 1

    # FIN
    if flags & 0x01:
        consumed_sequence += 1

    if consumed_sequence <= 0:
        return None

    start = int(tcp.seq)
    end = start + consumed_sequence

    return start, end, payload_length


def ranges_overlap(start_a, end_a, start_b, end_b):
    return start_a < end_b and start_b < end_a


def first_dns_answer_ttl(dns):
    if int(dns.ancount or 0) <= 0:
        return 0

    ttls = []

    current = dns.an

    for _ in range(int(dns.ancount or 0)):
        if current is None:
            break

        try:
            if hasattr(current, "ttl"):
                ttls.append(int(current.ttl))
        except (TypeError, ValueError):
            pass

        try:
            current = current.payload
        except AttributeError:
            break

    return min(ttls) if ttls else 0


def detect_l7_protocol(pkt, source_port, destination_port):
    # Mevcut extractor mapping'i korunuyor.
    # Eğitim verisindeki L7_PROTO ID sözlüğü ayrıca doğrulanmalı.

    if DNS in pkt:
        return 1

    if source_port == 21 or destination_port == 21:
        return 2

    if (
        source_port in {443, 8443}
        or destination_port in {443, 8443}
    ):
        return 3

    return 0

In [4]:
def create_flow(
    source_ip,
    destination_ip,
    source_port,
    destination_port,
    protocol,
    timestamp
):
    return {
        "source_ip": source_ip,
        "destination_ip": destination_ip,
        "source_port": int(source_port),
        "destination_port": int(destination_port),
        "protocol": int(protocol),

        "first_ts": float(timestamp),
        "last_ts": float(timestamp),

        "first_in_ts": None,
        "last_in_ts": None,

        "first_out_ts": None,
        "last_out_ts": None,

        "IN_BYTES": 0,
        "IN_PKTS": 0,

        "OUT_BYTES": 0,
        "OUT_PKTS": 0,

        "TCP_FLAGS": 0,
        "CLIENT_TCP_FLAGS": 0,
        "SERVER_TCP_FLAGS": 0,

        "MIN_TTL": None,
        "MAX_TTL": 0,

        "LONGEST_FLOW_PKT": 0,
        "SHORTEST_FLOW_PKT": None,

        "MIN_IP_PKT_LEN": None,
        "MAX_IP_PKT_LEN": 0,

        "NUM_PKTS_UP_TO_128_BYTES": 0,
        "NUM_PKTS_128_TO_256_BYTES": 0,
        "NUM_PKTS_256_TO_512_BYTES": 0,
        "NUM_PKTS_512_TO_1024_BYTES": 0,
        "NUM_PKTS_1024_TO_1514_BYTES": 0,

        "TCP_WIN_MAX_IN": 0,
        "TCP_WIN_MAX_OUT": 0,

        "RETRANSMITTED_IN_BYTES": 0,
        "RETRANSMITTED_IN_PKTS": 0,

        "RETRANSMITTED_OUT_BYTES": 0,
        "RETRANSMITTED_OUT_PKTS": 0,

        "seen_tcp_ranges_in": [],
        "seen_tcp_ranges_out": [],

        "ICMP_TYPE": 0,
        "ICMP_IPV4_TYPE": 0,

        "DNS_QUERY_ID": 0,
        "DNS_QUERY_TYPE": 0,
        "DNS_TTL_ANSWER": 0,

        "FTP_COMMAND_RET_CODE": 0,

        "L7_PROTO": 0
    }

In [5]:
def update_directional_statistics(
    flow,
    pkt,
    direction
):
    timestamp = float(pkt.time)

    # NF/IP byte sayımı için Ethernet frame değil,
    # IP paket uzunluğunu kullanıyoruz.
    ip_packet_length = int(
        pkt[IP].len or len(pkt[IP])
    )

    flow["last_ts"] = max(
        flow["last_ts"],
        timestamp
    )

    if direction == "in":
        flow["IN_BYTES"] += ip_packet_length
        flow["IN_PKTS"] += 1

        if flow["first_in_ts"] is None:
            flow["first_in_ts"] = timestamp

        flow["last_in_ts"] = timestamp

    else:
        flow["OUT_BYTES"] += ip_packet_length
        flow["OUT_PKTS"] += 1

        if flow["first_out_ts"] is None:
            flow["first_out_ts"] = timestamp

        flow["last_out_ts"] = timestamp


def update_length_statistics(flow, pkt):
    frame_length = len(pkt)

    ip_packet_length = int(
        pkt[IP].len or len(pkt[IP])
    )

    flow["LONGEST_FLOW_PKT"] = max(
        flow["LONGEST_FLOW_PKT"],
        frame_length
    )

    if flow["SHORTEST_FLOW_PKT"] is None:
        flow["SHORTEST_FLOW_PKT"] = frame_length
    else:
        flow["SHORTEST_FLOW_PKT"] = min(
            flow["SHORTEST_FLOW_PKT"],
            frame_length
        )

    flow["MAX_IP_PKT_LEN"] = max(
        flow["MAX_IP_PKT_LEN"],
        ip_packet_length
    )

    if flow["MIN_IP_PKT_LEN"] is None:
        flow["MIN_IP_PKT_LEN"] = ip_packet_length
    else:
        flow["MIN_IP_PKT_LEN"] = min(
            flow["MIN_IP_PKT_LEN"],
            ip_packet_length
        )

    ttl = int(pkt[IP].ttl)

    if flow["MIN_TTL"] is None:
        flow["MIN_TTL"] = ttl
    else:
        flow["MIN_TTL"] = min(
            flow["MIN_TTL"],
            ttl
        )

    flow["MAX_TTL"] = max(
        flow["MAX_TTL"],
        ttl
    )

    update_packet_size_bin(
        flow,
        frame_length
    )


def update_tcp_statistics(
    flow,
    pkt,
    direction
):
    if TCP not in pkt:
        return

    flags = int(pkt[TCP].flags)
    window = int(pkt[TCP].window or 0)

    flow["TCP_FLAGS"] |= flags

    if direction == "in":
        flow["CLIENT_TCP_FLAGS"] |= flags

        flow["TCP_WIN_MAX_IN"] = max(
            flow["TCP_WIN_MAX_IN"],
            window
        )

    else:
        flow["SERVER_TCP_FLAGS"] |= flags

        flow["TCP_WIN_MAX_OUT"] = max(
            flow["TCP_WIN_MAX_OUT"],
            window
        )


def update_retransmission_statistics(
    flow,
    pkt,
    direction
):
    sequence_info = tcp_sequence_range(pkt)

    if sequence_info is None:
        return

    start, end, payload_length = sequence_info

    ranges_key = (
        "seen_tcp_ranges_in"
        if direction == "in"
        else "seen_tcp_ranges_out"
    )

    is_retransmission = any(
        ranges_overlap(
            start,
            end,
            old_start,
            old_end
        )
        for old_start, old_end
        in flow[ranges_key]
    )

    if is_retransmission:
        if direction == "in":
            flow["RETRANSMITTED_IN_PKTS"] += 1
            flow["RETRANSMITTED_IN_BYTES"] += payload_length

        else:
            flow["RETRANSMITTED_OUT_PKTS"] += 1
            flow["RETRANSMITTED_OUT_BYTES"] += payload_length

    else:
        flow[ranges_key].append(
            (start, end)
        )


def update_dns_statistics(flow, pkt):
    if DNS not in pkt:
        return

    dns = pkt[DNS]

    flow["L7_PROTO"] = 1
    flow["DNS_QUERY_ID"] = int(dns.id or 0)

    if dns.qd is not None:
        try:
            question = dns.qd

            if isinstance(question, list):
                question = question[0]

            flow["DNS_QUERY_TYPE"] = int(
                question.qtype or 0
            )

        except (
            AttributeError,
            TypeError,
            ValueError,
            IndexError
        ):
            pass

    if int(dns.qr or 0) == 1:
        answer_ttl = first_dns_answer_ttl(dns)

        if answer_ttl > 0:
            flow["DNS_TTL_ANSWER"] = answer_ttl


def update_icmp_statistics(flow, pkt):
    if ICMP not in pkt:
        return

    icmp_type = int(
        pkt[ICMP].type or 0
    )

    flow["ICMP_TYPE"] = icmp_type
    flow["ICMP_IPV4_TYPE"] = icmp_type


def update_ftp_statistics(flow, pkt):
    if TCP not in pkt or Raw not in pkt:
        return

    source_port = int(pkt[TCP].sport)
    destination_port = int(pkt[TCP].dport)

    if (
        source_port != 21
        and destination_port != 21
    ):
        return

    flow["L7_PROTO"] = 2

    payload = bytes(pkt[Raw].load)

    match = FTP_REPLY_PATTERN.match(payload)

    if match:
        flow["FTP_COMMAND_RET_CODE"] = int(
            match.group(1)
        )

In [6]:
flows = {}

non_ipv4_count = 0

for pkt in packets:
    if IP not in pkt:
        non_ipv4_count += 1
        continue

    source_ip = pkt[IP].src
    destination_ip = pkt[IP].dst

    source_port, destination_port, protocol = (
        get_transport_info(pkt)
    )

    forward_key = make_forward_key(
        source_ip,
        destination_ip,
        source_port,
        destination_port,
        protocol
    )

    reverse_key = make_reverse_key(
        source_ip,
        destination_ip,
        source_port,
        destination_port,
        protocol
    )

    if forward_key in flows:
        flow_key = forward_key
        direction = "in"

    elif reverse_key in flows:
        flow_key = reverse_key
        direction = "out"

    else:
        # İlk görülen paket initiator/client yönüdür.
        flow_key = forward_key
        direction = "in"

        flows[flow_key] = create_flow(
            source_ip=source_ip,
            destination_ip=destination_ip,
            source_port=source_port,
            destination_port=destination_port,
            protocol=protocol,
            timestamp=float(pkt.time)
        )

    flow = flows[flow_key]

    update_directional_statistics(
        flow,
        pkt,
        direction
    )

    update_length_statistics(
        flow,
        pkt
    )

    update_tcp_statistics(
        flow,
        pkt,
        direction
    )

    update_retransmission_statistics(
        flow,
        pkt,
        direction
    )

    update_dns_statistics(
        flow,
        pkt
    )

    update_icmp_statistics(
        flow,
        pkt
    )

    update_ftp_statistics(
        flow,
        pkt
    )

    detected_l7 = detect_l7_protocol(
        pkt,
        source_port,
        destination_port
    )

    if flow["L7_PROTO"] == 0:
        flow["L7_PROTO"] = detected_l7

print("Okunan paket:", len(packets))
print("Atlanan IPv4 olmayan paket:", non_ipv4_count)
print("Olusturulan bidirectional flow:", len(flows))


Okunan paket: 365
Atlanan IPv4 olmayan paket: 33
Olusturulan bidirectional flow: 81


In [7]:
def finalize_flow(flow):
    flow_duration_ms = max(
        0.0,
        (
            flow["last_ts"]
            - flow["first_ts"]
        ) * 1000.0
    )

    duration_in_ms = directional_duration_ms(
        flow["first_in_ts"],
        flow["last_in_ts"]
    )

    duration_out_ms = directional_duration_ms(
        flow["first_out_ts"],
        flow["last_out_ts"]
    )

    src_second_bytes = calculate_second_bytes(
        flow["IN_BYTES"],
        flow["first_in_ts"],
        flow["last_in_ts"]
    )

    dst_second_bytes = calculate_second_bytes(
        flow["OUT_BYTES"],
        flow["first_out_ts"],
        flow["last_out_ts"]
    )

    src_avg_throughput = (
        src_second_bytes * 8.0
    )

    dst_avg_throughput = (
        dst_second_bytes * 8.0
    )

    return {
        "PROTOCOL": flow["protocol"],
        "L7_PROTO": flow["L7_PROTO"],

        "IN_BYTES": flow["IN_BYTES"],
        "IN_PKTS": flow["IN_PKTS"],

        "OUT_BYTES": flow["OUT_BYTES"],
        "OUT_PKTS": flow["OUT_PKTS"],

        "TCP_FLAGS": flow["TCP_FLAGS"],
        "CLIENT_TCP_FLAGS":
            flow["CLIENT_TCP_FLAGS"],
        "SERVER_TCP_FLAGS":
            flow["SERVER_TCP_FLAGS"],

        "FLOW_DURATION_MILLISECONDS":
            flow_duration_ms,

        "DURATION_IN": duration_in_ms,
        "DURATION_OUT": duration_out_ms,

        "MIN_TTL":
            flow["MIN_TTL"]
            if flow["MIN_TTL"] is not None
            else 0,

        "MAX_TTL": flow["MAX_TTL"],

        "LONGEST_FLOW_PKT":
            flow["LONGEST_FLOW_PKT"],

        "SHORTEST_FLOW_PKT":
            flow["SHORTEST_FLOW_PKT"]
            if flow["SHORTEST_FLOW_PKT"] is not None
            else 0,

        "MIN_IP_PKT_LEN":
            flow["MIN_IP_PKT_LEN"]
            if flow["MIN_IP_PKT_LEN"] is not None
            else 0,

        "MAX_IP_PKT_LEN":
            flow["MAX_IP_PKT_LEN"],

        "SRC_TO_DST_SECOND_BYTES":
            src_second_bytes,

        "DST_TO_SRC_SECOND_BYTES":
            dst_second_bytes,

        "RETRANSMITTED_IN_BYTES":
            flow["RETRANSMITTED_IN_BYTES"],

        "RETRANSMITTED_IN_PKTS":
            flow["RETRANSMITTED_IN_PKTS"],

        "RETRANSMITTED_OUT_BYTES":
            flow["RETRANSMITTED_OUT_BYTES"],

        "RETRANSMITTED_OUT_PKTS":
            flow["RETRANSMITTED_OUT_PKTS"],

        "SRC_TO_DST_AVG_THROUGHPUT":
            src_avg_throughput,

        "DST_TO_SRC_AVG_THROUGHPUT":
            dst_avg_throughput,

        "NUM_PKTS_UP_TO_128_BYTES":
            flow["NUM_PKTS_UP_TO_128_BYTES"],

        "NUM_PKTS_128_TO_256_BYTES":
            flow["NUM_PKTS_128_TO_256_BYTES"],

        "NUM_PKTS_256_TO_512_BYTES":
            flow["NUM_PKTS_256_TO_512_BYTES"],

        "NUM_PKTS_512_TO_1024_BYTES":
            flow["NUM_PKTS_512_TO_1024_BYTES"],

        "NUM_PKTS_1024_TO_1514_BYTES":
            flow["NUM_PKTS_1024_TO_1514_BYTES"],

        "TCP_WIN_MAX_IN":
            flow["TCP_WIN_MAX_IN"],

        "TCP_WIN_MAX_OUT":
            flow["TCP_WIN_MAX_OUT"],

        "ICMP_TYPE":
            flow["ICMP_TYPE"],

        "ICMP_IPV4_TYPE":
            flow["ICMP_IPV4_TYPE"],

        "DNS_QUERY_ID":
            flow["DNS_QUERY_ID"],

        "DNS_QUERY_TYPE":
            flow["DNS_QUERY_TYPE"],

        "DNS_TTL_ANSWER":
            flow["DNS_TTL_ANSWER"],

        "FTP_COMMAND_RET_CODE":
            flow["FTP_COMMAND_RET_CODE"]
    }


feature_rows = [
    finalize_flow(flow)
    for flow in flows.values()
]

features = pd.DataFrame(feature_rows)

print("Ham shape:", features.shape)
print(features.head())

Ham shape: (81, 39)
   PROTOCOL  L7_PROTO  IN_BYTES  IN_PKTS  OUT_BYTES  OUT_PKTS  TCP_FLAGS  \
0         6         0        84        2         84         2         24   
1         6         0       156        3          0         0          2   
2         6         0       156        3          0         0          2   
3         6         0        84        2         84         2         24   
4         6         0        84        2         84         2         24   

   CLIENT_TCP_FLAGS  SERVER_TCP_FLAGS  FLOW_DURATION_MILLISECONDS  ...  \
0                24                24                10388.576031  ...   
1                 2                 0                12018.074036  ...   
2                 2                 0                 6019.402981  ...   
3                24                24                10052.031040  ...   
4                24                24                 9392.276049  ...   

   NUM_PKTS_512_TO_1024_BYTES  NUM_PKTS_1024_TO_1514_BYTES  TCP_WIN_MAX_IN  \


In [8]:
missing_columns = [
    column
    for column in feature_names
    if column not in features.columns
]

extra_columns = [
    column
    for column in features.columns
    if column not in feature_names
]

print("Eksik kolonlar:", missing_columns)
print("Fazla kolonlar:", extra_columns)

assert not missing_columns, (
    f"Eksik kolon bulundu: {missing_columns}"
)

features = features[
    feature_names
].copy()

features = features.apply(
    pd.to_numeric,
    errors="coerce"
)

features = features.replace(
    [np.inf, -np.inf],
    np.nan
)

features = features.fillna(0.0)

FLOAT32_SAFE_LIMIT = (
    np.finfo(np.float32).max / 10.0
)

features = features.clip(
    lower=-FLOAT32_SAFE_LIMIT,
    upper=FLOAT32_SAFE_LIMIT
)

features = features.astype(
    np.float32
)

assert features.shape[1] == 39
assert features.isna().sum().sum() == 0
assert np.isfinite(
    features.to_numpy()
).all()

features.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nFeature CSV kaydedildi:")
print(OUTPUT_PATH.resolve())

print("\nShape:")
print(features.shape)

print("\nNaN sayisi:")
print(int(features.isna().sum().sum()))

print("\nTum degerler finite mi?")
print(
    np.isfinite(
        features.to_numpy()
    ).all()
)

Eksik kolonlar: []
Fazla kolonlar: []

Feature CSV kaydedildi:
C:\Users\egemen.keles\PycharmProjects\mitre_mapper\deneme_nfv2_features.csv

Shape:
(81, 39)

NaN sayisi:
0

Tum degerler finite mi?
True


In [10]:
probabilities = model.predict_proba(features)

top3_indices = np.argsort(
    probabilities,
    axis=1
)[:, -3:][:, ::-1]

report_rows = []

flow_keys = list(flows.keys())

for flow_index in range(len(features)):

    flow_key = flow_keys[flow_index]

    row = {
        "flow_index": flow_index,

        "src_ip": flow_key[0],
        "dst_ip": flow_key[1],

        "src_port": flow_key[2],
        "dst_port": flow_key[3],

        "protocol": flow_key[4],

        # add full key
        "flow_key": str(flow_key)
    }

    row_probs = probabilities[flow_index]

    row["prob_sum"] = float(
        row_probs.sum()
    )

    row["max_probability"] = float(
        np.max(row_probs)
    )

    for rank, class_index in enumerate(
        top3_indices[flow_index],
        start=1
    ):

        label = label_encoder.inverse_transform(
            np.array(
                [class_index],
                dtype=np.int32
            )
        )[0]

        probability = probabilities[
            flow_index,
            class_index
        ]

        row[f"top_{rank}_label"] = label
        row[f"top_{rank}_probability"] = float(
            probability
        )

    report_rows.append(row)

prediction_report = pd.DataFrame(
    report_rows
)

prediction_report = prediction_report.sort_values(
    "max_probability",
    ascending=False
)

print(
    prediction_report.to_string(
        index=False
    )
)

 flow_index          src_ip          dst_ip  src_port  dst_port  protocol                                          flow_key  prob_sum  max_probability    top_1_label  top_1_probability    top_2_label  top_2_probability    top_3_label  top_3_probability
         18      10.34.2.36      10.36.0.30     49542        53        17       ('10.34.2.36', '10.36.0.30', 49542, 53, 17)       1.0         0.998284         Benign           0.998284   Infiltration           0.000697 Reconnaissance           0.000398
          1      10.34.2.36      10.5.89.53     54306      7680         6      ('10.34.2.36', '10.5.89.53', 54306, 7680, 6)       1.0         0.997747         Benign           0.997747           DDoS           0.000755            Bot           0.000488
          2      10.34.2.36     10.5.32.161     54311      7680         6     ('10.34.2.36', '10.5.32.161', 54311, 7680, 6)       1.0         0.997148         Benign           0.997148 Reconnaissance           0.000982           DDoS        

In [12]:
bot_flows = prediction_report[
    prediction_report["top_1_label"] == "Bot"
]

print(bot_flows[
    [
        "src_ip",
        "dst_ip",
        "src_port",
        "dst_port",
        "protocol",
        "top_1_probability"
    ]
].to_string(index=False))

      src_ip     dst_ip  src_port  dst_port  protocol  top_1_probability
10.35.15.144 10.34.2.36      7680     63764         6           0.758256
  10.5.28.54 10.34.2.36     62004      7680         6           0.745398
  10.41.8.10 10.34.2.36     51754      7680         6           0.745386


In [13]:
import numpy as np
import pandas as pd


# ============================================================
# 1. Keep prediction rows aligned with feature rows
# ============================================================

debug_df = prediction_report.copy().reset_index(drop=True)

feature_debug = features.copy().reset_index(drop=True)
feature_debug.columns = [
    f"feature__{column}"
    for column in feature_debug.columns
]

debug_df = pd.concat(
    [
        debug_df,
        feature_debug
    ],
    axis=1
)


# ============================================================
# 2. Select port-7680 traffic
# ============================================================

port_7680_mask = (
    debug_df["src_port"].eq(7680)
    | debug_df["dst_port"].eq(7680)
)

port_7680_df = debug_df.loc[
    port_7680_mask
].copy()


# ============================================================
# 3. Separate false Bot and benign examples
# ============================================================

bot_7680 = port_7680_df.loc[
    port_7680_df["top_1_label"]
    .astype(str)
    .str.strip()
    .eq("Bot")
].copy()

benign_7680 = port_7680_df.loc[
    port_7680_df["top_1_label"]
    .astype(str)
    .str.strip()
    .eq("Benign")
].copy()


print("Port 7680 flows:", len(port_7680_df))
print("Bot predictions:", len(bot_7680))
print("Benign predictions:", len(benign_7680))


# ============================================================
# 4. Display false-positive Bot flows
# ============================================================

identity_columns = [
    "flow_index",
    "src_ip",
    "dst_ip",
    "src_port",
    "dst_port",
    "protocol",
    "top_1_label",
    "top_1_probability",
    "top_2_label",
    "top_2_probability",
]

print("\nFALSE BOT PREDICTIONS:")
print(
    bot_7680[
        identity_columns
    ].to_string(index=False)
)


# ============================================================
# 5. Print their complete 39-feature vectors
# ============================================================

model_feature_columns = [
    column
    for column in debug_df.columns
    if column.startswith("feature__")
]

print("\nFALSE BOT FEATURE VECTORS:")

for _, row in bot_7680.iterrows():

    flow_name = (
        f"{row['src_ip']}:{int(row['src_port'])}"
        f" -> "
        f"{row['dst_ip']}:{int(row['dst_port'])}"
    )

    print("\n" + "=" * 100)
    print("FLOW:", flow_name)
    print("BOT PROBABILITY:", row["top_1_probability"])
    print("=" * 100)

    feature_values = row[
        model_feature_columns
    ].copy()

    feature_values.index = [
        column.replace("feature__", "", 1)
        for column in feature_values.index
    ]

    print(feature_values.to_string())


# ============================================================
# 6. Check whether Bot feature vectors are duplicates
# ============================================================

bot_feature_matrix = bot_7680[
    model_feature_columns
].copy()

duplicate_mask = bot_feature_matrix.duplicated(
    keep=False
)

print(
    "\nExact duplicate Bot vectors:",
    int(duplicate_mask.sum()),
    "/",
    len(bot_feature_matrix)
)

if duplicate_mask.any():
    print("\nDUPLICATE BOT VECTORS:")
    print(
        bot_7680.loc[
            duplicate_mask,
            identity_columns
        ].to_string(index=False)
    )


# ============================================================
# 7. Find features that differ between Bot and Benign 7680
# ============================================================

if not bot_7680.empty and not benign_7680.empty:

    bot_numeric = (
        bot_7680[model_feature_columns]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    benign_numeric = (
        benign_7680[model_feature_columns]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    comparison_rows = []

    for column in model_feature_columns:

        bot_median = bot_numeric[column].median()
        benign_median = benign_numeric[column].median()

        bot_mean = bot_numeric[column].mean()
        benign_mean = benign_numeric[column].mean()

        absolute_median_difference = abs(
            bot_median - benign_median
        )

        denominator = max(
            abs(benign_median),
            1e-9
        )

        relative_median_difference = (
            absolute_median_difference
            / denominator
        )

        comparison_rows.append(
            {
                "feature": column.replace(
                    "feature__",
                    "",
                    1
                ),
                "bot_median": bot_median,
                "benign_median": benign_median,
                "bot_mean": bot_mean,
                "benign_mean": benign_mean,
                "absolute_median_difference":
                    absolute_median_difference,
                "relative_median_difference":
                    relative_median_difference,
            }
        )

    comparison_df = pd.DataFrame(
        comparison_rows
    )

    comparison_df = comparison_df.sort_values(
        [
            "relative_median_difference",
            "absolute_median_difference",
        ],
        ascending=False
    )

    print(
        "\nTOP DIFFERENCES BETWEEN BOT AND "
        "BENIGN PORT-7680 FLOWS:"
    )

    print(
        comparison_df.head(20).to_string(
            index=False
        )
    )

else:
    print(
        "\nComparison skipped because Bot or Benign "
        "port-7680 examples are missing."
    )

Port 7680 flows: 66
Bot predictions: 3
Benign predictions: 63

FALSE BOT PREDICTIONS:
 flow_index       src_ip     dst_ip  src_port  dst_port  protocol top_1_label  top_1_probability top_2_label  top_2_probability
          4 10.35.15.144 10.34.2.36      7680     63764         6         Bot           0.758256      Benign           0.238433
          0   10.5.28.54 10.34.2.36     62004      7680         6         Bot           0.745398      Benign           0.251136
          3   10.41.8.10 10.34.2.36     51754      7680         6         Bot           0.745386      Benign           0.251132

FALSE BOT FEATURE VECTORS:

FLOW: 10.35.15.144:7680 -> 10.34.2.36:63764
BOT PROBABILITY: 0.7582563161849976
PROTOCOL                             6.0
L7_PROTO                             0.0
IN_BYTES                            44.0
IN_PKTS                              1.0
OUT_BYTES                           40.0
OUT_PKTS                             1.0
TCP_FLAGS                           24.0
CLIENT